In [15]:
%cd /home/anw2067/visualnav-transformer/train

import torch
import yaml
import copy
import wandb
import json

import numpy as np
from torchvision import transforms
from dreamsim import dreamsim
from scipy.spatial.transform import Rotation as R

from diffusers.models import AutoencoderKL

from peva.models import CDiT_models
from peva.diffusion import create_diffusion

from vint_train.training.nymeria_training_utils import forward_kinematics_wrapper
from vint_train.training.nymeria_training_utils import unnormalize_data_smpl_pose_gaussian
from vint_train.models.nomad.nomad_vint import NoMaD_ViNT, replace_bn_with_gn
from vint_train.models.nomad.conditional_uned1dnomad import ConditionalUnet1D_NoMaD
from vint_train.models.nomad.nomad import DenseNetwork, NoMaD
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from vint_train.training.nymeria_training_utils import normalize_data_smpl_pose
from vint_train.data.misc import XSensConstants, XsensSkeleton
from vint_train.data.vint_dataset import ViNT_Nymeria_Dataset
from vint_train.training.nymeria_training_utils import get_action_smpl_torch
from planning.cem import CEMPlanner

from torchvision.utils import save_image

PEVA_CONFIG="/home/anw2067/visualnav-transformer/train/peva/config/nymeria_rel_concat_embedding_compile_beta095_ar_model_context_16_bs_16_smpl_lowebody_-64to_64_1_goal_emb_relative_xxl.yaml"
PEVA_CHECKPOINT="/scratch/anw2067/nymeria_rel_concat_embedding_compile_beta095_ar_model_context_16_bs_16_smpl_lowebody_cancel_scaler_-64to_64_xxl_280_0180000.pth.tar"

NOMAD_CONFIG = "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2025_12_09_11_24:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw/config.yaml"
NOMAD_CHECKPOINT = "/home/anw2067/visualnav-transformer/train/logs/nomad-minimal/2025_12_09_11_24:nomad-minimal-proprioception-cat8-dinov3_unpool_3dposemb-proj-lr5e-4-pool_curr_obs-goaldraw/ema_9.pth"


Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f8b0057ae60, raw_cell="%cd /home/anw2067/visualnav-transformer/train

imp.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#W0sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

/home/anw2067/visualnav-transformer/train
Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f8b0057ace0, execution_count=15 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f8b0057ae60, raw_cell="%cd /home/anw2067/visualnav-transformer/train

imp.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#W0sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [16]:
def load_peva(peva_config_file, peva_checkpoint, diffusion_steps=250, inference_context_size=15, device='cpu'):
    with open(peva_config_file, 'r') as f:
        peva_config = yaml.safe_load(f)
    num_cond = peva_config['context_size']
    if inference_context_size is None:
        inference_context_size = num_cond
    
    if peva_config.get('full_body', False):
        num_head_joint_cond = 23
    else:
        num_head_joint_cond = 15
        
    model = CDiT_models[peva_config['model']](
        context_size=num_cond, 
        inference_context_size=inference_context_size,
        num_head_joint_cond=num_head_joint_cond, 
        input_size=peva_config['image_size'] // 8, 
        in_channels=4, 
        diffusion_forcing=peva_config.get('diffusion_forcing', 0), 
        is_eval=1, skip_action_embedding=peva_config.get('skip_action_embedding', True), total_feature_dim=peva_config.get('total_feature_dim', None)).to(device)
    
    try:
        model = torch.compile(model)
        print("Compiled model")
    except:
        print("Failed to compile model")
        model = model

    try:
        ckp = torch.load(peva_checkpoint, map_location='cpu', weights_only=False)
        model.load_state_dict(ckp["ema"], strict=True)
    except:
        print(f"Checkpoint {peva_checkpoint} not found. Running with random init.")
        
    model.eval()

    diffusion = create_diffusion(str(diffusion_steps))
    vae = AutoencoderKL.from_pretrained(f"stabilityai/sd-vae-ft-ema").to(device)
    # model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[device])
    # model_without_ddp = model.module
    model_without_ddp = model
    
    peva_stats = {"min": torch.tensor([-2, -1, -1, -1, -1, -1], dtype=torch.float32, device=device)[None], # 1,  6
                  "max": torch.tensor([2, 1, 1, 1, 1, 1], dtype=torch.float32, device=device)[None]} # 1, 6
    
    return model, model_without_ddp, diffusion, vae, peva_stats, peva_config

model, _, diffusion, vae, peva_stats, peva_config = load_peva(PEVA_CONFIG, PEVA_CHECKPOINT, device='cuda')

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f8b0059af50, raw_cell="def load_peva(peva_config_file, peva_checkpoint, d.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#W1sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

Compiled model
Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f8b0059b430, execution_count=16 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f8b0059af50, raw_cell="def load_peva(peva_config_file, peva_checkpoint, d.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#W1sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [17]:
def load_policy(nomad_config_file, nomad_checkpoint, device='cpu'):
    with open(nomad_config_file, "r") as f:
        config = yaml.safe_load(f)
        
    def get_vision_encoder():
        if config.get("goal_type", None) in ["2d", "2d5050"]:
            goal_coordinate_dims = 8
        else:
            goal_coordinate_dims = 0
        vision_encoder = NoMaD_ViNT(
            obs_encoder=config["obs_encoder"],
            obs_encoding_size=config["encoding_size"],
            context_size=config["context_size"],
            mha_num_attention_heads=config["mha_num_attention_heads"],
            mha_num_attention_layers=config["mha_num_attention_layers"],
            mha_ff_dim_factor=config["mha_ff_dim_factor"],
            pool_features=config.get("pool_features", True),
            image_size=config["image_size"],
            proprioception=config.get("proprioception", False),
            project_encoding=config.get("project_encoding", False),
            pos_enc_3d=config.get("pos_enc_3d", False),
            pool_curr_obs=config.get("pool_curr_obs", False),
            goal_coordinate_dims=goal_coordinate_dims,
        )
        vision_encoder = replace_bn_with_gn(vision_encoder)
        return vision_encoder
    
    vision_encoder = get_vision_encoder()
        
    if config.get("goal_type", None) == "cheat":
        goal_pose_dim = 48
    elif config.get("goal_type", None) == "point":
        goal_pose_dim = 4 * 3 # 3 dimensions each for (Head, LHand, RHand, Pelvis)
    else:
        goal_pose_dim = 0
    noise_pred_net = ConditionalUnet1D_NoMaD(input_dim=config['input_dims'],
                                                global_cond_dim=config["encoding_size"],
                                                down_dims=config["down_dims"],
                                                cond_predict_scale=config["cond_predict_scale"],
                                                goal_pose_dims=goal_pose_dim)
    dist_pred_network = DenseNetwork(embedding_dim=config["encoding_size"])
    model = NoMaD(vision_encoder, noise_pred_net, dist_pred_network)
    noise_scheduler = DDPMScheduler(num_train_timesteps=config["num_diffusion_iters"], beta_schedule='squaredcos_cap_v2', clip_sample=True, prediction_type='epsilon')
    
    model = model.to(device)
    model = model.eval()
    noise_scheduler = noise_scheduler

    loaded_state_dict = torch.load(nomad_checkpoint, map_location=device) 
    for key in list(loaded_state_dict.keys()):
        if "module" in key:
            loaded_state_dict[key.replace("module.", "")] = loaded_state_dict[key]
            del loaded_state_dict[key] # remove the module prefix
            
    res = model.load_state_dict(loaded_state_dict, strict=True)
    print("model loaded with: ", res)

    with open(config['datasets']['nymeria']['gaussian_normalization_stats_path'], 'r') as f:
        stats_json = json.load(f)
    stats_dict = {"mean": stats_json['pelvis_xyz']['mean'], "var": stats_json['pelvis_xyz']['var']}
    for part_name in XSensConstants.part_names[:XSensConstants.upper_body_num_parts]:
        stats_dict["mean"] += stats_json['rpy'][part_name]['mean']
        stats_dict["var"] += stats_json['rpy'][part_name]['var']
                
    nomad_stats = {
        "mean": torch.tensor(stats_dict["mean"], dtype=torch.float32)[None, None], # 1, 1, 48
        "var": torch.tensor(stats_dict["var"], dtype=torch.float32)[None, None] # 1, 1, 48
    }
    
    return model, noise_scheduler, nomad_stats, config

policy, noise_scheduler, nomad_stats, config = load_policy(NOMAD_CONFIG, NOMAD_CHECKPOINT, device='cuda')

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f871c676d10, raw_cell="def load_policy(nomad_config_file, nomad_checkpoin.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#W2sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


model loaded with:  <All keys matched successfully>
Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f871c675600, execution_count=17 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f871c676d10, raw_cell="def load_policy(nomad_config_file, nomad_checkpoin.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#W2sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [18]:
def get_nymeria_dataset(config, context_size=16-2, split="test"):
    data_config = config["datasets"]["nymeria"]
    
    if "waypoint_spacing" not in data_config:
        data_config["waypoint_spacing"] = 1
    if "negative_goals" not in data_config:
        data_config["negative_goals"] = False
    if "end_slack" not in data_config:
        data_config["end_slack"] = 0
    if "goals_per_obs" not in data_config:
        data_config["goals_per_obs"] = 1

    ### EVAL ONLY -- DO NOT NORMALIZE THE DELTAS
    data_config["normalize"] = False
    
    # transform = ([
    #     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    # ])
    # transform = transforms.Compose(transform)
    transform = torch.nn.Identity()
    
    if context_size is None:
        context_size = config["context_size"]
    
    dataset = ViNT_Nymeria_Dataset(
        data_folder=data_config["data_folder"],
        data_split_folder=data_config[split],
        dataset_name="nymeria",
        image_size=config["image_size"],
        transform=transform,
        waypoint_spacing=data_config["waypoint_spacing"],
        min_dist_cat=config["distance"]["min_dist_cat"],
        max_dist_cat=config["distance"]["max_dist_cat"],
        min_action_distance=config["action"]["min_dist_cat"],
        max_action_distance=config["action"]["max_dist_cat"],
        negative_goals=data_config["negative_goals"],
        len_traj_pred=config["len_traj_pred"],
        context_size=context_size,
        goal_type=config.get("goal_type", None),
        preserve_pose_up_down=data_config.get("preserve_pose_up_down", False),
        end_slack=data_config["end_slack"],
        goals_per_obs=data_config["goals_per_obs"],
        normalize=config["normalize"],
        gaussian_normalization_stats_path=data_config["gaussian_normalization_stats_path"],
    )
    return dataset

nomad_config = yaml.load(open(NOMAD_CONFIG), Loader=yaml.FullLoader)
dataset = get_nymeria_dataset(nomad_config)

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f871c6edc30, raw_cell="def get_nymeria_dataset(config, context_size=16-2,.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#W3sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f871c6ee290, execution_count=18 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f871c6edc30, raw_cell="def get_nymeria_dataset(config, context_size=16-2,.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#W3sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [19]:
@torch.no_grad()
def model_forward_wrapper(all_models, curr_obs, curr_delta, latent_size, device, num_cond, rel_t=None, progress=False):
    model, diffusion, vae = all_models
    x = curr_obs.to(device)
    y = curr_delta.to(device)
    
    with torch.amp.autocast('cuda', enabled=True, dtype=torch.bfloat16):
        B, T = x.shape[:2]

        if rel_t is None:
            raise ValueError("Not implemented")

        x = x.flatten(0,1)
        y = y.flatten(0, 1)
        rel_t = rel_t.flatten(0, 1)
        
        x = vae.encode(x).latent_dist.sample().mul_(0.18215).unflatten(0, (B, T))
        x_cond = x[:, :num_cond]
        z = torch.randn(x.shape[0], 4, latent_size, latent_size, device=device)
        t_cond = torch.zeros(x.shape[0], x_cond.shape[1], device=device)
        model_kwargs = dict(y=y, rel_t=rel_t, num_cond=num_cond, x_cond=x_cond, t_cond=t_cond, x_clean=x.flatten(0, 1))
        samples = diffusion.p_sample_loop(model.forward, z.shape, z, clip_denoised=False, model_kwargs=model_kwargs, progress=progress, device=device) # B, 4, latent_size, latent_size; samples a single image
        samples = vae.decode(samples / 0.18215).sample # B, 3, image_size, image_size 
        return torch.clip(samples, -1., 1.)

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f871c3a1c90, raw_cell="@torch.no_grad()
def model_forward_wrapper(all_mod.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#W4sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f871c695c60, execution_count=19 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f871c3a1c90, raw_cell="@torch.no_grad()
def model_forward_wrapper(all_mod.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#W4sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [20]:
@torch.no_grad()
def policy_sample(
    model: torch.nn.Module,
    noise_scheduler: DDPMScheduler,
    batch_obs_images: torch.Tensor,
    batch_goal_images: torch.Tensor, # or None
    context_poses: torch.Tensor,
    pred_horizon: int,
    action_dim: int,
    device: torch.device,
):
    """
    Generate model output (conditioned, unconditioned, distance) for the given batch of images.
    Outputs are DELTAS and are NOT unnormalized or scaled.
    """
    N = batch_obs_images.shape[0]
    timesteps = noise_scheduler.timesteps.to(device)
    
    # assume goal is always provided.
    no_mask = torch.zeros((batch_goal_images.shape[0],), device=device).long()
    obsgoal_cond = model("vision_encoder",
                         obs_img=batch_obs_images,
                         goal_img=batch_goal_images, 
                         input_goal_mask=no_mask,
                         context_poses=context_poses, goal_coordinates=None)

    # initialize action from Gaussian noise
    noisy_diffusion_output = torch.randn(
        (N, pred_horizon, action_dim), device=device)
    diffusion_output = noisy_diffusion_output

    # initialize action from Gaussian noise
    noisy_diffusion_output = torch.randn(
        (N, pred_horizon, action_dim), device=device)
    diffusion_output = noisy_diffusion_output

    for k in timesteps:
        # predict noise
        noise_pred = model(
            "noise_pred_net",
            sample=diffusion_output,
            timestep=k.unsqueeze(-1).repeat(diffusion_output.shape[0]),
            global_cond=obsgoal_cond,
            goal_pose=None
        )

        # inverse diffusion step (remove noise)
        diffusion_output = noise_scheduler.step(
            model_output=noise_pred,
            timestep=k,
            sample=diffusion_output
        ).prev_sample
        
    diffusion_output = unnormalize_data_smpl_pose_gaussian(diffusion_output.flatten(0, 1)).unflatten(0, (N, pred_horizon))
    return diffusion_output

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f871c6743a0, raw_cell="@torch.no_grad()
def policy_sample(
    model: tor.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#W5sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f871c675600, execution_count=20 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f871c6743a0, raw_cell="@torch.no_grad()
def policy_sample(
    model: tor.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#W5sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [21]:
def _compute_pose_and_loss(actn, gt_actn, skel, actn_mask=None):
        gt_xyz, gt_rpy = forward_kinematics_wrapper(gt_actn, skel, XSensConstants.upper_body_num_parts, return_euler=True) # B, num_segments, 3
        pred_xyz, pred_rpy = forward_kinematics_wrapper(actn, skel, XSensConstants.upper_body_num_parts, return_euler=True) # B, num_segments, 3
        res = {}
        for i, body_part_name in enumerate(XSensConstants.part_names[:XSensConstants.upper_body_num_parts]):
            R_gt = R.from_euler('xyz', gt_rpy[:, i, :].detach().cpu().numpy(), degrees=False)
            R_pred = R.from_euler('xyz', pred_rpy[:, i, :].detach().cpu().numpy(), degrees=False)
            ang_dist = torch.from_numpy((R_gt.inv() * R_pred).magnitude() / np.pi * 180).to(actn.device).float() # B
            xyz_dist = torch.norm(gt_xyz[:, i, :] - pred_xyz[:, i, :], dim=-1) # B
            if actn_mask is not None:
                ang_dist = ang_dist * actn_mask
                xyz_dist = xyz_dist * actn_mask
            res[f"{body_part_name}-angular_distance"] = ang_dist
            res[f"{body_part_name}-xyz_distance"] = xyz_dist
        return res

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f870f773e20, raw_cell="def _compute_pose_and_loss(actn, gt_actn, skel, ac.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#W6sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f870f773d30, execution_count=21 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f870f773e20, raw_cell="def _compute_pose_and_loss(actn, gt_actn, skel, ac.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#W6sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [22]:
from torchvision.utils import draw_keypoints
def draw_waypoints(obs, waypoints, color_order=["red", "green", "blue", "yellow"]):
    """
    Draws waypoint as circles on an image
    
    Args:
        obs: B, 3, H, W 
        waypoints: B, 8 or B, 4, 2
        color_order: list of colors
    """
    
    if waypoints.shape[1] == 8:
        waypoints = waypoints.reshape(*waypoints.shape[:-1], 4, 2)
        
    B = obs.shape[0]
    for b in range(B):
        for index, color in enumerate(color_order):
            obs[b] = draw_keypoints(obs[b], waypoints[b, index:index+1, None, :], colors=color, radius=4)
    return obs
    

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f8713e88eb0, raw_cell="from torchvision.utils import draw_keypoints
def d.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#X10sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f8713e8b550, execution_count=22 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f8713e88eb0, raw_cell="from torchvision.utils import draw_keypoints
def d.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#X10sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [23]:
def waypoint_sample(policy_model, policy_diffusion,
                    peva_model, peva_diffusion, peva_vae, peva_stats,
                    waypoints, context_poses, curr_obs, goal_obs,
                    policy_pred_horizon, policy_action_dim,
                    image_size, 
                    policy_context_size, peva_context_size, peva_latent_size,
                    device,
                    skip_optional_peva=False):
    imagenet_norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    wm_norm = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    
    B, W = waypoints.shape[:2]
    delta_accum = torch.zeros(B, W * policy_pred_horizon, policy_action_dim, device=device) # B, W*8, 48
    generated_frames = torch.zeros(B, W * policy_pred_horizon, 3, image_size, image_size, device=device)
    for w in range(W):
        policy_obs = imagenet_norm(curr_obs[:, -policy_context_size:].flatten(0, 1)).unflatten(0, (B, policy_context_size))
        goal_obs = imagenet_norm(draw_waypoints(curr_obs[:, -1], waypoints[:, w:w+1]))
        deltas = policy_sample(policy_model, policy_diffusion,
                    policy_obs, goal_obs,
                    context_poses[:, :policy_context_size], 
                    policy_pred_horizon, policy_action_dim, device) # B, 8, 48
        delta_accum[:, w * policy_pred_horizon:(w+1) * policy_pred_horizon] = deltas
    
        if skip_optional_peva and w == W-1:
            continue 
        
        peva_normalized_deltas = normalize_data_smpl_pose(deltas, peva_stats)
        for t in range(deltas.shape[1]):
            x_cond = torch.zeros(curr_obs.shape[0], peva_context_size+1, curr_obs.shape[2], curr_obs.shape[3], curr_obs.shape[4], device=device)
            x_cond[:, :peva_context_size] = wm_norm(curr_obs[:, -peva_context_size:].flatten(0, 1)).unflatten(0, (B, peva_context_size))
            
            curr_delta = peva_normalized_deltas[:, t:t+1].repeat(1, curr_obs.shape[1], 1,) 
            device = curr_obs.device
            
            rel_const = 1. / (64-(-64))  # distance is set in eval, but fixed to [8, 8] for now.
            rel_const = rel_const * 1 # multiply the rel_const by rollout_stride 
            rel_t = (torch.ones(curr_obs.shape[0], peva_context_size, device=device) * rel_const)
            
            x_pred = model_forward_wrapper(
                (peva_model, peva_diffusion, peva_vae),
                x_cond,
                curr_delta,
                peva_latent_size,
                device,
                num_cond=curr_obs.shape[1],
                rel_t=rel_t,
                progress=True
            )
            x_pred = x_pred[:, None] # B, 1, 3, H, W
            generated_frames[:, (w * policy_pred_horizon) + t] = x_pred[:, 0]
            curr_obs = torch.cat([curr_obs[:, 1:], x_pred], dim=1)
    return generated_frames, delta_accum

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f8713d30250, raw_cell="def waypoint_sample(policy_model, policy_diffusion.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#X11sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f8713d302b0, execution_count=23 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7f8713d30250, raw_cell="def waypoint_sample(policy_model, policy_diffusion.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#X11sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [ ]:
class Preprocessor:
    def __init__(self, transform):
        self.transform = transform

    def transform_obs(self, obs):
        res = {}
        for key in obs:
            if key == "images":
                res[key] = self.transform(obs[key])
            elif key == "goal_image":
                res[key] = self.transform(obs[key])
            else:
                res[key] = obs[key]
        return res

class WaypointWM(torch.nn.Module):
    def __init__(self,peva_model, peva_diffusion, peva_vae, peva_stats,
                 policy_model, policy_diffusion,
                 image_size, peva_context_size, 
                 policy_context_size, policy_pred_horizon, policy_action_dim):
        super().__init__()
        self.peva_model = peva_model
        self.peva_diffusion = peva_diffusion
        self.peva_vae = peva_vae
        self.peva_stats = peva_stats
        
        self.policy_model = policy_model
        self.policy_diffusion = policy_diffusion
        
        self.image_size = image_size
        self.latent_size = image_size // 8
        self.peva_context_size = peva_context_size
        self.policy_context_size = policy_context_size
        
        self.policy_pred_horizon = policy_pred_horizon
        self.policy_action_dim = policy_action_dim
        
    def encode_obs(self, obs):
        return copy.deepcopy(obs)
    
    def rollout(self, obs_0, act):
        device = act.device
        curr_obs = obs_0['images'] # B, 15, 3, H, W
        goal_obs = obs_0['goal_image'] # B, 3, H, W
        context_poses = obs_0['context_poses'] # B, 15, 48
        
        generated_frames, delta_accum = waypoint_sample(self.policy_model, self.policy_diffusion,
                    self.peva_model, self.peva_diffusion, self.peva_vae, self.peva_stats,
                    act, context_poses, curr_obs, goal_obs,
                    self.policy_pred_horizon, self.policy_action_dim,
                    self.image_size, 
                    self.policy_context_size, self.peva_context_size, self.latent_size,
                    device)
        
        # all_frames = torch.cat([obs_0['images'], generated_frames], dim=1)
        # all_frames = all_frames * 0.5 + 0.5
        # for i in range(all_frames.shape[0]):
        #     image = torch.cat([img for img in all_frames[i]], dim=-1)
        #     save_image(image, f"rollout_{i}.png")
        return {"images": generated_frames.to(torch.float32), "deltas": delta_accum.to(torch.float32)}, None

class ObjectiveFn:
    def __init__(self, device):
        self.device = device
        self.model, self.preprocess = dreamsim(pretrained=True, device=device, cache_dir="/scratch/anw2067/cache")
        
    def __call__(self, rollout_state, goal_state):
        
        pred_image = rollout_state["images"]
        goal_image = goal_state["images"]
        B = pred_image.shape[0]
        res = []
        for i in range(B):
            pred_image_pil = transforms.ToPILImage()(pred_image[i, -1])
            goal_image_pil = transforms.ToPILImage()(goal_image[i])
            
            rollout_state = self.preprocess(pred_image_pil).to(self.device)
            goal_state = self.preprocess(goal_image_pil).to(self.device)

            sim = self.model(rollout_state, goal_state)
            res.append(sim)
        res = torch.cat(res, dim=0) # B
        print(f"ObjectiveFn: {res.mean().item()}")
        return res

class Evaluator(WaypointWM):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
    
    def eval_actions(self, actions_mu, state_0, state_g):
        """
        actions_mu: B, T, action_dim
        gt_dict: dict of gt_actions, skel
        """
        waypoints = actions_mu 
        curr_obs = state_0['images'] # B, 15, 3, H, W
        goal_obs = state_0['goal_image'] # B, 3, H, W
        context_poses = state_0['context_poses'] # B, 15, 48
        device = curr_obs.device
        
        _, pred_actions = waypoint_sample(self.policy_model, self.policy_diffusion,
                    self.peva_model, self.peva_diffusion, self.peva_vae, self.peva_stats,
                    waypoints, context_poses, curr_obs, goal_obs,
                    self.policy_pred_horizon, self.policy_action_dim,
                    self.image_size, 
                    self.policy_context_size, self.peva_context_size, self.latent_size,
                    device,
                    skip_optional_peva=True)
        
        deltas_gt = state_g["deltas"]
        first_pose = state_g["first_pose"] # B, 1, 48
        xsens_offsets = state_g["xsens_offsets"]
        skel = XsensSkeleton(xsens_offsets)
        
        pred_actions = get_action_smpl_torch(first_pose, pred_actions, XSensConstants.upper_body_num_parts) # B, T, 48
        gt_actions = get_action_smpl_torch(first_pose, deltas_gt, XSensConstants.upper_body_num_parts) # B, T, 48
        eval_metrics = _compute_pose_and_loss(pred_actions[:, -1], gt_actions[:, -1], skel)
        
        start_distances = _compute_pose_and_loss(first_pose[:, -1], gt_actions[:, -1], skel)
        leaf_start_distances = {k: v for k, v in start_distances.items() if any(x in k.lower() for x in ["head", "hand", "pelvis"])}
        leaf_start_avg_xyz_distance = sum([v.mean().item() for k, v in leaf_start_distances.items() if "xyz" in k.lower()]) / len(leaf_start_distances)
        leaf_start_avg_angular_distance = sum([v.mean().item() for k, v in leaf_start_distances.items() if "angular" in k.lower()]) / len(leaf_start_distances)
        print(f"start-leaf-xyz_distance: {leaf_start_avg_xyz_distance}")
        print(f"start-leaf-angular_distance: {leaf_start_avg_angular_distance}")
        
        res = {}
        for k, v in eval_metrics.items():
            if any(x in k.lower() for x in ["head", "hand" "pelvis"]):
                leaf_key = "leaf-" + k.split("-")[1]
                if leaf_key not in res: res[leaf_key] = []
                res[leaf_key].append(v)
        for k, v in res.items():
            res[k] = torch.cat(v, dim=0).mean().item()
            print(f"{k}: {res[k]}")
        for k, v in eval_metrics.items():
            res[k] = v.mean().item()
        res["start-leaf-xyz_distance"] = leaf_start_avg_xyz_distance
        res["start-leaf-angular_distance"] = leaf_start_avg_angular_distance
        for k, v in res.items():
            print(f"{k}: {v}")

        return res
        
wm_wrapper = WaypointWM(model, diffusion, vae, peva_stats, policy, noise_scheduler,
                nomad_config["image_size"][0], peva_config["context_size"], nomad_config["context_size"]+1,
                nomad_config["len_traj_pred"], nomad_config["input_dims"])   

evaluator = Evaluator(model, diffusion, vae, peva_stats, policy, noise_scheduler,
                nomad_config["image_size"][0], peva_config["context_size"], nomad_config["context_size"]+1,
                nomad_config["len_traj_pred"], nomad_config["input_dims"])
     
cem_planner = CEMPlanner(
    horizon=1,
    topk=2,
    num_samples=4,
    var_scale=0.5,
    opt_steps=8,
    eval_every=1,
    wm=wm_wrapper,
    action_dim=8,
    objective_fn=ObjectiveFn(device="cuda"),
    preprocessor=Preprocessor(transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])),
    evaluator=evaluator,
    wandb_run=None,
    logging_prefix="waypoint_cem-4top1-var1"
)

wandb.init(project="peva-planning", name="cem")

batch = dataset[0]

obs_images = batch["obs_images"][None] # 1, context_size, 3, H, W
goal_image = batch["goal_image"][None] # 1, 3, H, W
context_poses = batch["context_poses"][None] # 1, context_size, 48

deltas = batch["deltas"][None] # 1, horizon, action_dim
first_pose = batch["first_pose"][None] # 1, 1, 48
xsens_offsets = batch["xsens_offsets"] # 15, 3
goal_obs = batch["goal_obs"][None] # 1, 3, H, W

obs_0 = {"images": obs_images, "goal_image": goal_image, "context_poses": context_poses}
obs_g = {"images": goal_obs, "deltas": deltas, "first_pose": first_pose, "xsens_offsets": xsens_offsets}

cem_planner.plan(obs_0, obs_g)

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7f8b0057bd90>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7f871c695c60, raw_cell="class Preprocessor:
    def __init__(self, transfo.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22746f726368767363227d/home/anw2067/visualnav-transformer/train/notebooks/waypoint_cem.ipynb#X12sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

Using cached /scratch/anw2067/cache


Using cache found in /scratch/anw2067/cache/facebookresearch_dino_main
/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


/scratch/anw2067/conda/envs/nomad_train2/lib/python3.10/site-packages/wandb/sdk/internal/internal_api.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


  7%|▋         | 17/250 [00:01<00:19, 12.19it/s]